In [ ]:
# Ô CODE 1 — Cấu hình screening và Seed 43
import gc
import math
import random
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

SEED = 43
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = SEED) -> None:
    """Khóa các nguồn ngẫu nhiên của Python, NumPy và PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# Cấu hình vĩ mô dùng chung.
GAMMA = 0.95
NN_LR = 5e-5
VAE_LR = 1e-3
COST_LR = 1e-3
ZETA = 0.05
BETA_0_LIST = [0.01, 0.03, 0.05, 0.10, 0.15, 0.30, 0.50]
BETA_DECAY = 0.91
BETA_MIN = 0.01
EPISODES = 45
SARSA_ALPHA = 0.60
Q_WEIGHT_DECAY = 1e-4
VAE_BETA_KL = 0.01
LATENT_DIM = 16
BATCH_SIZE = 128
VAE_BATCH_SIZE = 256
COST_BATCH_SIZE = 256
BOOTSTRAP_TRAJECTORIES = 5
BOOTSTRAP_UPDATES = 100
ONLINE_AUX_UPDATES = 1
REPLAY_CAPACITY = 50_000
BALANCE_INIT = 1_000.0
TRANSACTION_FEE = 0.001
W_RISK = 0.15
W_STABILITY = 0.05
ACTION_VALUES = np.arange(-5, 6, dtype=np.int64)
RISK_FREE_RATE_PERCENT = 2.0

VARIANTS = {
    "Legacy Ablation": {
        "robust_loss": False, "weight_decay": 0.0, "reward_shaping": False,
        "kl_reduction": "mean", "vae_beta_kl": 1e-3, "use_cost": False,
    },
    "Enhanced UCB-VAE": {
        "robust_loss": True, "weight_decay": Q_WEIGHT_DECAY, "reward_shaping": True,
        "kl_reduction": "sum", "vae_beta_kl": VAE_BETA_KL, "use_cost": False,
    },
    "Uncertainty-Aware UCB-VAE": {
        "robust_loss": True, "weight_decay": Q_WEIGHT_DECAY, "reward_shaping": True,
        "kl_reduction": "sum", "vae_beta_kl": VAE_BETA_KL, "use_cost": True,
    },
}

print({"seed": SEED, "device": str(DEVICE), "beta_grid": BETA_0_LIST, "zeta": ZETA})


In [ ]:
# Ô CODE 2 — Dữ liệu HPG BAD, môi trường và Frozen Standardization
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL"), *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "data_storer" / "data_research").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/data_storer/data_research.")


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "data_storer" / "data_research"
TRAIN_CSV = DATA_ROOT / "train" / "bad_train_HPG.csv"
TEST_CSV = DATA_ROOT / "test" / "bad_test_HPG.csv"
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "kaggle_working"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_COLUMNS = ["time", "close", "MACD", "RSI", "CCI", "ADX"]
STATE_NAMES = ["Price", "Balance", "Position", "MACD", "RSI", "CCI", "ADX"]


def load_hpg_bad() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train, test = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    for label, frame in (("train", train), ("test", test)):
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing:
            raise ValueError(f"HPG BAD {label} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
        numeric = frame[REQUIRED_COLUMNS[1:]].apply(pd.to_numeric, errors="coerce")
        if numeric.isna().any().any():
            raise ValueError(f"HPG BAD {label} chứa NaN/giá trị không hợp lệ.")
        frame[REQUIRED_COLUMNS[1:]] = numeric
    if train["time"].max() >= test["time"].min():
        raise ValueError("Train/Test chồng lấn thời gian; dừng để tránh leakage.")
    return train.reset_index(drop=True), test.reset_index(drop=True)


def raw_state(row: pd.Series, cash: float, position: int) -> np.ndarray:
    return np.asarray(
        [row["close"], cash, position, row["MACD"], row["RSI"], row["CCI"], row["ADX"]],
        dtype=np.float32,
    )


class TradingEnv:
    """Single-asset, long-only; action là số cổ phiếu giao dịch trong [-5, 5]."""
    def __init__(self, data: pd.DataFrame, reward_shaping: bool, initial_cash: float = BALANCE_INIT):
        if len(data) < 2:
            raise ValueError("TradingEnv cần ít nhất hai quan sát.")
        self.data = data.reset_index(drop=True)
        self.reward_shaping = bool(reward_shaping)
        self.initial_cash = float(initial_cash)
        self.reset()

    def reset(self) -> np.ndarray:
        self.index, self.cash, self.position = 0, self.initial_cash, 0
        self.peak_value = self.initial_cash
        self.portfolio_history = [self.initial_cash]
        self.var_targets: List[float] = []
        return self._state()

    def _state(self) -> np.ndarray:
        return raw_state(self.data.iloc[self.index], self.cash, self.position)

    def _execute(self, requested: int, price: float) -> int:
        if requested > 0:
            affordable = int(self.cash // (price * (1.0 + TRANSACTION_FEE)))
            executed = min(int(requested), affordable)
        else:
            executed = -min(-int(requested), self.position)
        traded_value = abs(executed) * price
        self.cash -= executed * price + traded_value * TRANSACTION_FEE
        self.position += executed
        return executed

    def step(self, action: int):
        current_price = float(self.data.iloc[self.index]["close"])
        previous_value = self.cash + self.position * current_price
        executed = self._execute(int(action), current_price)
        self.index += 1
        next_price = float(self.data.iloc[self.index]["close"])
        portfolio_value = self.cash + self.position * next_price
        raw_profit = portfolio_value - previous_value
        portfolio_return = raw_profit / max(abs(previous_value), 1e-8)
        var_target = max(-portfolio_return, 0.0)
        self.peak_value = max(self.peak_value, portfolio_value)
        drawdown = (self.peak_value - portfolio_value) / max(self.peak_value, 1e-8)
        asset_return = next_price / max(current_price, 1e-8) - 1.0
        reward = raw_profit
        if self.reward_shaping:
            reward -= W_RISK * abs(drawdown) + W_STABILITY * abs(asset_return)
        self.portfolio_history.append(float(portfolio_value))
        self.var_targets.append(float(var_target))
        done = self.index >= len(self.data) - 1
        info = {
            "raw_profit": float(raw_profit), "portfolio_return": float(portfolio_return),
            "var_target": float(var_target), "drawdown": float(drawdown),
            "executed_action": int(executed), "portfolio_value": float(portfolio_value),
        }
        return self._state(), float(reward), done, info


@dataclass(frozen=True)
class FrozenScaler:
    mean: np.ndarray
    std: np.ndarray

    def transform(self, states: np.ndarray) -> np.ndarray:
        x = np.asarray(states, dtype=np.float32)
        return (x - self.mean) / self.std


def train_only_calibration_states(train: pd.DataFrame, trajectories: int = 5) -> np.ndarray:
    """Ước lượng đủ 7 chiều state bằng random rollout chỉ trên tập train."""
    rng = np.random.default_rng(SEED)
    states: List[np.ndarray] = []
    for _ in range(trajectories):
        env = TradingEnv(train, reward_shaping=False)
        state, done = env.reset(), False
        while not done:
            states.append(state.copy())
            action = int(rng.choice(ACTION_VALUES))
            state, _, done, _ = env.step(action)
    return np.asarray(states, dtype=np.float32)


train_hpg, test_hpg = load_hpg_bad()
calibration = train_only_calibration_states(train_hpg)
frozen_mean = calibration.mean(axis=0, dtype=np.float64).astype(np.float32)
frozen_std = np.maximum(calibration.std(axis=0, dtype=np.float64), 1e-6).astype(np.float32)
FROZEN_SCALER = FrozenScaler(frozen_mean.copy(), frozen_std.copy())
FROZEN_SCALER.mean.setflags(write=False)
FROZEN_SCALER.std.setflags(write=False)
print(pd.DataFrame({"feature": STATE_NAMES, "train_mean": frozen_mean, "train_std": frozen_std}).to_string(index=False))
print({"train": len(train_hpg), "test": len(test_hpg), "train_end": train_hpg.time.max(), "test_start": test_hpg.time.min()})


In [ ]:
# Ô CODE 3 — Q-Network, EpistemicVAE và Cost Network
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states)


class EpistemicVAE(nn.Module):
    """Joint VAE: 7 state + 11 action one-hot → latent 16 → reconstruction 18."""
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 18))

    def encode(self, x: torch.Tensor):
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        reconstructed = self.decoder(mu + torch.randn_like(std) * std)
        return reconstructed, mu, logvar

    def compute_u_ep(self, states, actions_onehot, kl_reduction: str = "sum"):
        """Novelty với KLD sum/mean tùy ablation và probe z⁺=μ+1.96σ."""
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        kl_terms = -0.5 * (1.0 + logvar - mu.square() - logvar.exp())
        if kl_reduction == "sum":
            kl = torch.sum(kl_terms, dim=-1)
        elif kl_reduction == "mean":
            kl = torch.mean(kl_terms, dim=-1)
        else:
            raise ValueError("kl_reduction phải là 'sum' hoặc 'mean'.")
        std = torch.exp(0.5 * logvar)
        reconstructed_95 = self.decoder(mu + 1.96 * std)
        reconstruction_error = torch.norm(x - reconstructed_95, p=2, dim=-1)
        return kl + reconstruction_error


class CostNetwork(nn.Module):
    """Ước lượng VaR không âm từ concat(s_scaled, action_onehot) ∈ R^18."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([states, actions_onehot], dim=-1)).squeeze(-1)


def onehot(action_indices: torch.Tensor) -> torch.Tensor:
    return F.one_hot(action_indices.long(), num_classes=11).to(dtype=torch.float32)


print(QNetwork(), "\n", EpistemicVAE(), "\n", CostNetwork())


In [ ]:
# Ô CODE 4 — Loss, replay và cơ chế Confidence-Weighted Risk Penalty
def reconstruction_loss(prediction, target, robust: bool):
    return F.huber_loss(prediction, target) if robust else F.mse_loss(prediction, target)


def vae_loss(reconstructed, target, mu, logvar, beta_kl: float, robust: bool):
    recon = reconstruction_loss(reconstructed, target, robust)
    kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
    return recon + float(beta_kl) * kl


def cost_loss(predicted_var, var_target):
    return F.huber_loss(predicted_var, var_target)


class ReplayBuffer:
    def __init__(self, capacity: int = REPLAY_CAPACITY):
        self.capacity = int(capacity)
        self.states: List[np.ndarray] = []
        self.actions: List[int] = []
        self.var_targets: List[float] = []

    def add(self, states, action_indices, var_targets):
        for state, action, var_target in zip(states, action_indices, var_targets):
            self.states.append(np.asarray(state, dtype=np.float32))
            self.actions.append(int(action))
            self.var_targets.append(float(var_target))
        overflow = len(self.states) - self.capacity
        if overflow > 0:
            del self.states[:overflow]
            del self.actions[:overflow]
            del self.var_targets[:overflow]

    def sample(self, batch_size: int):
        size = min(int(batch_size), len(self.states))
        if size == 0:
            raise RuntimeError("Không thể sample replay buffer rỗng.")
        indices = np.random.choice(len(self.states), size=size, replace=False)
        return (
            np.asarray([self.states[i] for i in indices]),
            np.asarray([self.actions[i] for i in indices]),
            np.asarray([self.var_targets[i] for i in indices], dtype=np.float32),
        )

    def __len__(self):
        return len(self.states)


def action_scores(q_network, vae, cost_network, state, beta, config, collect_details=False):
    """Q - confidence·VaR + beta·novelty_scaled theo đúng đặc tả."""
    scaled = FROZEN_SCALER.transform(np.asarray(state).reshape(1, -1))
    state_tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
    state_batch = state_tensor.repeat(11, 1)
    action_batch = torch.eye(11, dtype=torch.float32, device=DEVICE)
    q_network.eval(); vae.eval()
    if cost_network is not None:
        cost_network.eval()
    with torch.no_grad():
        q_values = q_network(state_tensor).squeeze(0)
        novelty_raw = vae.compute_u_ep(state_batch, action_batch, config["kl_reduction"])
        novelty_scaled = novelty_raw / (novelty_raw.max() + 1e-8)
        calibrated_penalty = torch.zeros_like(q_values)
        predicted_var = torch.zeros_like(q_values)
        if config["use_cost"]:
            predicted_var = cost_network(state_batch, action_batch)
            confidence = 1.0 / (1.0 + novelty_raw.clamp_min(0.0))
            calibrated_penalty = torch.where(
                predicted_var < ZETA,
                torch.zeros_like(predicted_var),
                confidence * predicted_var,
            )
        scores = q_values - calibrated_penalty + float(beta) * novelty_scaled
        action_index = int(torch.argmax(scores).item())
    details = None
    if collect_details:
        details = {
            "novelty": novelty_raw.detach().cpu().numpy(),
            "predicted_var": predicted_var.detach().cpu().numpy(),
            "penalty": calibrated_penalty.detach().cpu().numpy(),
        }
    return int(ACTION_VALUES[action_index]), action_index, details


def auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config):
    raw_states, action_indices, var_targets = replay.sample(VAE_BATCH_SIZE)
    states = torch.as_tensor(FROZEN_SCALER.transform(raw_states), dtype=torch.float32, device=DEVICE)
    indices = torch.as_tensor(action_indices, dtype=torch.long, device=DEVICE)
    actions = onehot(indices)
    reconstructed, mu, logvar = vae(states, actions)
    target = torch.cat([states, actions], dim=-1)
    loss_vae = vae_loss(reconstructed, target, mu, logvar, config["vae_beta_kl"], config["robust_loss"])
    vae_optimizer.zero_grad(set_to_none=True)
    loss_vae.backward()
    torch.nn.utils.clip_grad_norm_(vae.parameters(), 5.0)
    vae_optimizer.step()

    loss_cost_value = np.nan
    if config["use_cost"]:
        targets = torch.as_tensor(var_targets, dtype=torch.float32, device=DEVICE)
        predicted = cost_network(states, actions)
        loss_c = cost_loss(predicted, targets)
        cost_optimizer.zero_grad(set_to_none=True)
        loss_c.backward()
        torch.nn.utils.clip_grad_norm_(cost_network.parameters(), 5.0)
        cost_optimizer.step()
        loss_cost_value = float(loss_c.detach().cpu())
    return float(loss_vae.detach().cpu()), loss_cost_value


def random_bootstrap(replay: ReplayBuffer):
    rng = np.random.default_rng(SEED)
    for _ in range(BOOTSTRAP_TRAJECTORIES):
        env = TradingEnv(train_hpg, reward_shaping=False)
        state, done = env.reset(), False
        states, actions, var_targets = [], [], []
        while not done:
            action_index = int(rng.integers(0, 11))
            states.append(state.copy()); actions.append(action_index)
            state, _, done, info = env.step(int(ACTION_VALUES[action_index]))
            var_targets.append(info["var_target"])
        replay.add(states, actions, var_targets)


def collect_episode(env, q_network, vae, cost_network, beta, config):
    state, done = env.reset(), False
    states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index, _ = action_scores(q_network, vae, cost_network, state, beta, config)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy())
        rewards.append(reward); actions.append(action_index); dones.append(done)
        var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets


def q_update(q_network, optimizer, trajectory, robust: bool):
    states, next_states, rewards, actions, next_actions, dones, _ = trajectory
    s = torch.as_tensor(FROZEN_SCALER.transform(states), dtype=torch.float32, device=DEVICE)
    sn = torch.as_tensor(FROZEN_SCALER.transform(next_states), dtype=torch.float32, device=DEVICE)
    r = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(actions, dtype=torch.long, device=DEVICE)
    an = torch.as_tensor(next_actions, dtype=torch.long, device=DEVICE)
    terminal = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE)
    losses: List[float] = []
    q_network.train()
    for start in range(0, len(states), BATCH_SIZE):
        sl = slice(start, min(start + BATCH_SIZE, len(states)))
        current = q_network(s[sl]).gather(1, a[sl, None]).squeeze(1)
        with torch.no_grad():
            following = q_network(sn[sl]).gather(1, an[sl, None]).squeeze(1)
            td_target = r[sl] + GAMMA * (1.0 - terminal[sl]) * following
            target = (1.0 - SARSA_ALPHA) * current.detach() + SARSA_ALPHA * td_target
        loss = F.huber_loss(current, target) if robust else F.mse_loss(current, target)
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(q_network.parameters(), 5.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    return losses


In [ ]:
# Ô CODE 5 — Coarse screening: 3 biến thể × 7 beta × 45 episodes × Seed 43
def evaluation_frame(data: pd.DataFrame, previous_row: Optional[pd.Series] = None) -> pd.DataFrame:
    return data if previous_row is None else pd.concat([previous_row.to_frame().T, data], ignore_index=True)


def evaluate(q_network, vae, cost_network, beta, config, data, previous_row=None, collect_diagnostics=False):
    env = TradingEnv(evaluation_frame(data, previous_row), reward_shaping=config["reward_shaping"])
    state, done = env.reset(), False
    novelty_raw: List[float] = []
    predicted_var: List[float] = []
    calibrated_penalty: List[float] = []
    while not done:
        action, action_index, details = action_scores(
            q_network, vae, cost_network, state, beta, config, collect_details=collect_diagnostics
        )
        if details is not None:
            novelty_raw.extend(details["novelty"].tolist())  # Raw u_ep của đủ 11 action.
            predicted_var.append(float(details["predicted_var"][action_index]))
            calibrated_penalty.append(float(details["penalty"][action_index]))
        state, _, done, _ = env.step(action)
    diagnostics = {
        "novelty_raw": np.asarray(novelty_raw, dtype=np.float64),
        "predicted_var": np.asarray(predicted_var, dtype=np.float64),
        "calibrated_penalty": np.asarray(calibrated_penalty, dtype=np.float64),
    }
    return np.asarray(env.portfolio_history), np.asarray(env.var_targets), diagnostics


def metrics(portfolio: np.ndarray, var_targets: np.ndarray) -> Dict[str, float]:
    profit = float(portfolio[-1] - portfolio[0])
    years = max((test_hpg.time.iloc[-1] - test_hpg.time.iloc[0]).days / 365.25, 1 / 365.25)
    ratio = max(float(portfolio[-1] / max(portfolio[0], 1e-8)), 1e-12)
    arr = (ratio ** (1.0 / years) - 1.0) * 100.0
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    annual_return = np.mean(returns) * 252.0 * 100.0 if len(returns) else 0.0
    volatility = np.std(returns) * np.sqrt(252.0) * 100.0 if len(returns) else 0.0
    sharpe = 0.0 if volatility < 1e-12 else (annual_return - RISK_FREE_RATE_PERCENT) / volatility
    peaks = np.maximum.accumulate(portfolio)
    max_drawdown = abs(np.min((portfolio - peaks) / np.maximum(peaks, 1e-8))) * 100.0
    violations = int(np.sum(var_targets > ZETA))
    return {
        "Final Profit": profit, "ARR (%)": float(arr), "Sharpe Ratio": float(sharpe),
        "Max Drawdown (%)": float(max_drawdown), "Constraint Violations": violations,
    }


def run_variant(variant_name: str, beta_0: float) -> Dict[str, object]:
    if variant_name not in VARIANTS:
        raise KeyError(f"Biến thể không hợp lệ: {variant_name}")
    config = dict(VARIANTS[variant_name])
    set_seed(SEED)  # Common random numbers cho từng cấu hình đối chứng.
    q_network, vae = QNetwork().to(DEVICE), EpistemicVAE().to(DEVICE)
    # Luôn khởi tạo CostNetwork để mọi biến thể tiêu thụ cùng RNG; chỉ UA mới sử dụng/cập nhật nó.
    cost_network = CostNetwork().to(DEVICE)
    q_optimizer = torch.optim.Adam(q_network.parameters(), lr=NN_LR, weight_decay=config["weight_decay"])
    vae_optimizer = torch.optim.Adam(vae.parameters(), lr=VAE_LR)
    cost_optimizer = torch.optim.Adam(cost_network.parameters(), lr=COST_LR) if config["use_cost"] else None
    replay = ReplayBuffer(); random_bootstrap(replay)
    q_losses: List[float] = []
    vae_losses: List[float] = []
    cost_losses: List[float] = []
    train_curve: List[float] = []
    test_curve: List[float] = []

    for _ in range(BOOTSTRAP_UPDATES):
        lv, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
        vae_losses.append(lv)
        if np.isfinite(lc): cost_losses.append(lc)

    for episode in range(EPISODES):
        beta = max(BETA_MIN, float(beta_0) * (BETA_DECAY ** episode))
        env = TradingEnv(train_hpg, reward_shaping=config["reward_shaping"])
        trajectory = collect_episode(env, q_network, vae, cost_network, beta, config)
        replay.add(trajectory[0], trajectory[3], trajectory[6])
        q_losses.extend(q_update(q_network, q_optimizer, trajectory, config["robust_loss"]))
        for _ in range(ONLINE_AUX_UPDATES):
            lv, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
            vae_losses.append(lv)
            if np.isfinite(lc): cost_losses.append(lc)
        train_portfolio, _, _ = evaluate(
            q_network, vae, cost_network, beta, config, train_hpg
        )
        train_curve.append(float(train_portfolio[-1] - BALANCE_INIT))
        test_portfolio, _, _ = evaluate(
            q_network, vae, cost_network, beta, config, test_hpg, train_hpg.iloc[-1]
        )
        test_curve.append(float(test_portfolio[-1] - BALANCE_INIT))

    final_beta = max(BETA_MIN, float(beta_0) * (BETA_DECAY ** (EPISODES - 1)))
    portfolio, var_targets, diagnostics = evaluate(
        q_network, vae, cost_network, final_beta, config, test_hpg, train_hpg.iloc[-1],
        collect_diagnostics=True,
    )
    result = {
        "Variant": variant_name, "beta_0": float(beta_0), "final_beta": final_beta,
        **metrics(portfolio, var_targets), "q_losses": q_losses,
        "vae_losses": vae_losses, "cost_losses": cost_losses,
        "train_curve": train_curve, "test_curve": test_curve,
        "realized_var": np.asarray(var_targets, dtype=np.float64), **diagnostics,
    }
    if not all(np.all(np.isfinite(result[key])) for key in ["q_losses", "vae_losses"]):
        raise FloatingPointError(f"Loss NaN/Inf ở {variant_name}, beta={beta_0}.")
    if config["use_cost"] and not np.all(np.isfinite(cost_losses)):
        raise FloatingPointError(f"Cost loss NaN/Inf ở {variant_name}, beta={beta_0}.")
    del q_network, vae, cost_network, q_optimizer, vae_optimizer, cost_optimizer, replay
    return result


RUN_SCREENING = True
experiment_runs: List[Dict[str, object]] = []
if RUN_SCREENING:
    started = time.perf_counter()
    total = len(VARIANTS) * len(BETA_0_LIST)
    for run_index, variant_name in enumerate(VARIANTS, start=0):
        for beta_index, beta_0 in enumerate(BETA_0_LIST, start=1):
            ordinal = run_index * len(BETA_0_LIST) + beta_index
            try:
                result = run_variant(variant_name, beta_0)
            except Exception as error:
                print(traceback.format_exc())
                raise RuntimeError(f"Thất bại tại {variant_name}, beta={beta_0}") from error
            experiment_runs.append(result)
            print(
                f"[{ordinal}/{total}] {variant_name} | beta={beta_0:.2f} | "
                f"profit={result['Final Profit']:.4f} | sharpe={result['Sharpe Ratio']:.4f} | "
                f"violations={result['Constraint Violations']}"
            )
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"Hoàn tất 9 cấu hình sau {time.perf_counter() - started:.1f}s")
else:
    print("RUN_SCREENING=False: chưa chạy coarse screening.")


In [ ]:
# Ô CODE 6 — Bảng Markdown, sensitivity, kiểm soát rủi ro và convergence
if not experiment_runs:
    raise RuntimeError("Chưa có kết quả. Hãy đặt RUN_SCREENING=True và chạy lại Ô CODE 5.")

metric_columns = [
    "Variant", "beta_0", "final_beta", "Final Profit", "ARR (%)",
    "Sharpe Ratio", "Max Drawdown (%)", "Constraint Violations",
]
results_df = pd.DataFrame([{key: run[key] for key in metric_columns} for run in experiment_runs])
results_df = results_df.sort_values(["Variant", "beta_0"]).reset_index(drop=True)
results_path = OUTPUT_DIR / "cost_network_three_variant_screening_seed43.csv"
results_df.to_csv(results_path, index=False)
print("### Battle of 3 Variants — HPG BAD Coarse Screening, Seed 43")
try:
    print(results_df.to_markdown(index=False, floatfmt=".6f"))
except ImportError:
    print(results_df.to_string(index=False))
    print("Cài 'tabulate' nếu muốn render bảng Markdown.")
print("Saved:", results_path)

# Chọn beta tốt nhất riêng cho từng biến thể theo Sharpe, tie-break bằng Profit.
best_runs = {}
for variant_name in VARIANTS:
    candidates = [run for run in experiment_runs if run["Variant"] == variant_name]
    best_runs[variant_name] = max(candidates, key=lambda run: (run["Sharpe Ratio"], run["Final Profit"]))
best_df = pd.DataFrame([{key: run[key] for key in metric_columns} for run in best_runs.values()])
print("\n### Best beta của từng biến thể")
try:
    print(best_df.to_markdown(index=False, floatfmt=".6f"))
except ImportError:
    print(best_df.to_string(index=False))

# Heuristic chữ U ngược cho Sharpe: beta giữa phải tốt hơn hai biên.
sensitivity_diagnostics = []
for variant_name in VARIANTS:
    subset = results_df[results_df["Variant"] == variant_name].sort_values("beta_0")
    values = subset["Sharpe Ratio"].to_numpy()
    peak_index = int(np.argmax(values))
    sensitivity_diagnostics.append({
        "Variant": variant_name,
        "Best beta": float(subset.iloc[peak_index]["beta_0"]),
        "Interior peak": bool(0 < peak_index < len(values) - 1),
    })
print("\n### Sensitivity diagnostics")
print(pd.DataFrame(sensitivity_diagnostics).to_string(index=False))

# 1) Sensitivity: hai panel cùng beta grid để không trộn đơn vị Profit và Sharpe.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for variant_name in VARIANTS:
    subset = results_df[results_df["Variant"] == variant_name].sort_values("beta_0")
    axes[0].plot(subset["beta_0"], subset["Final Profit"], marker="o", label=variant_name)
    axes[1].plot(subset["beta_0"], subset["Sharpe Ratio"], marker="o", label=variant_name)
axes[0].set(title="Sensitivity — Final Profit", xlabel="BETA_0", ylabel="Final Profit")
axes[1].set(title="Sensitivity — Sharpe Ratio", xlabel="BETA_0", ylabel="Sharpe Ratio")
for ax in axes: ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# 2) Risk control tại beta tốt nhất: MDD và số vi phạm tách panel.
labels = list(best_runs)
mdd = [best_runs[name]["Max Drawdown (%)"] for name in labels]
violations = [best_runs[name]["Constraint Violations"] for name in labels]
short_labels = ["Legacy", "Enhanced", "Uncertainty-Aware"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(short_labels, mdd, color=["tab:grey", "tab:blue", "tab:green"])
axes[1].bar(short_labels, violations, color=["tab:grey", "tab:blue", "tab:green"])
axes[0].set(title="Max Drawdown tại best beta", ylabel="Max Drawdown (%)")
axes[1].set(title=f"Constraint Violations (VaR > {ZETA:.0%})", ylabel="Số bước vi phạm")
for ax in axes: ax.grid(axis="y", alpha=0.25); ax.tick_params(axis="x", rotation=12)
plt.tight_layout(); plt.show()

# 3) Screening diagnostics của Uncertainty-Aware tốt nhất.
ua_run = best_runs["Uncertainty-Aware UCB-VAE"]

def smooth(values: Sequence[float], window: int = 10) -> np.ndarray:
    array = np.asarray(values, dtype=np.float64)
    if len(array) < window: return array
    return pd.Series(array).rolling(window, min_periods=1).mean().to_numpy()


fig, axes = plt.subplots(2, 2, figsize=(14, 10))
episode_axis = np.arange(1, EPISODES + 1)
axes[0, 0].plot(episode_axis, ua_run["train_curve"], color="tab:blue", label="Train profit")
axes[0, 0].plot(episode_axis, ua_run["test_curve"], color="tab:purple", label="Test profit")
axes[0, 0].fill_between(episode_axis, ua_run["train_curve"], ua_run["test_curve"], alpha=0.12, label="Generalization gap")
axes[0, 0].set(title="Train/Test Generalization Gap", xlabel="Episode", ylabel="Final Profit")
for key, label, color in [
    ("q_losses", "Q loss", "tab:blue"),
    ("vae_losses", "VAE loss", "tab:orange"),
    ("cost_losses", "Cost loss", "tab:green"),
]:
    values = smooth(ua_run[key])
    if len(values):
        progress = np.linspace(0.0, 1.0, len(values))
        axes[0, 1].plot(progress, np.maximum(values, 1e-12), label=label, color=color)
axes[0, 1].set(title="Loss Convergence (moving average)", xlabel="Normalized training progress", ylabel="Loss")
axes[0, 1].set_yscale("log")
finite_novelty = np.asarray(ua_run["novelty_raw"])
finite_novelty = finite_novelty[np.isfinite(finite_novelty)]
axes[1, 0].hist(finite_novelty, bins=40, color="tab:orange", alpha=0.8, label="u_ep raw")
axes[1, 0].set(title="Raw Novelty Distribution", xlabel="u_ep raw", ylabel="Frequency")
predicted = np.asarray(ua_run["predicted_var"])
realized = np.asarray(ua_run["realized_var"])
axes[1, 1].plot(smooth(predicted, 10), label="Predicted VaR", alpha=0.9)
axes[1, 1].plot(smooth(realized, 10), label="Realized VaR", alpha=0.8)
axes[1, 1].axhline(ZETA, color="red", linestyle="--", label=f"zeta={ZETA:.2f}")
axes[1, 1].set(title="Cost Network Calibration", xlabel="Test step", ylabel="VaR")
for ax in axes.ravel(): ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.show()

print({
    "ua_best_beta": ua_run["beta_0"],
    "q_loss_finite": bool(np.all(np.isfinite(ua_run["q_losses"]))),
    "vae_loss_finite": bool(np.all(np.isfinite(ua_run["vae_losses"]))),
    "cost_loss_finite": bool(np.all(np.isfinite(ua_run["cost_losses"]))),
    "caution": "Đây là đối chứng một seed; chưa đủ để suy luận ý nghĩa thống kê.",
})
